<a href="https://colab.research.google.com/github/sangjkim930/AI-Driven-Research-Methodology/blob/main/03_Reusable_Research_Knowledge_Base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building a Reusable Research Knowledge Base

**5 Papers → Vector Store → Ask → Add 1 Paper → Ask Again → Evidence File**

Run the cells in order from top to bottom.

### Step 1. Set Up the Environment

Connect to the OpenAI API and choose the model used for the research tasks.

In [4]:
!pip install -q openai

from openai import OpenAI
from google.colab import userdata, files

client = OpenAI(
    api_key=userdata.get("OPENAI_API_KEY")
)

ANSWER_MODEL = "gpt-5.6-luna"

### Step 2. Create the Research Knowledge Base

Upload four papers, create a vector store, and add the papers to it.

In [3]:
uploaded = files.upload()
paper_files = list(uploaded.keys())

vector_store = client.vector_stores.create(
    name="Research Knowledge Base"
)

for file_name in paper_files:
    client.vector_stores.files.upload_and_poll(
        vector_store_id=vector_store.id,
        file=open(file_name, "rb")
    )

print("Papers added:", len(paper_files))
print("Vector Store ID:", vector_store.id)

Saving 01_Boakye et al. (2020).pdf to 01_Boakye et al. (2020).pdf
Saving 02_Gallego-Alvarez et al. (2015).pdf to 02_Gallego-Alvarez et al. (2015).pdf
Saving 03_Fujii et al. (2013).pdf to 03_Fujii et al. (2013).pdf
Saving 04_Misani and Pogutz (2015).pdf to 04_Misani and Pogutz (2015).pdf
Saving 05_Muhammad et al. (2015).pdf to 05_Muhammad et al. (2015).pdf
Papers added: 5
Vector Store ID: vs_6a77f1f74f9c8191b5aefc9d1119aa01


### Step 3. Ask a Research Question

Ask a question using only the papers stored in the vector store.

In [5]:
QUESTION = """
Compare the relationship between environmental performance
and financial performance across the papers.

Use only the papers in the research knowledge base.
Distinguish each paper's own empirical findings from
prior studies cited in the paper.
If the evidence is insufficient, say so.
"""

response = client.responses.create(
    model=ANSWER_MODEL,
    input=QUESTION,
    tools=[
        {
            "type": "file_search",
            "vector_store_ids": [vector_store.id]
        }
    ]
)

print(response.output_text)

## Overall comparison

Across the papers, the evidence does **not** support a single, uniformly positive linear relationship between environmental and financial performance. The papers’ **own empirical results** are mostly favorable to environmental improvement, but they also show that the relationship depends on:

- the environmental measure used—carbon emissions, toxic-risk exposure, or broader environmental practices;
- whether performance is measured by accounting outcomes or market valuation;
- the intensity of environmental engagement;
- the business cycle and institutional context; and
- the distinction between environmental **outcomes** and environmental **processes**.

### Each paper’s own empirical findings

| Paper | Empirical setting and measures | Findings of the paper itself |
|---|---|---|
| **Boakye et al. (2020)** | 201 listed UK SMEs, 2011–2016; several environmental-practice measures and financial performance, principally ROA | Finds a **significant nonlinear, genera

### Step 4. Add One New Paper

Upload one additional paper and add it to the same vector store.

In [6]:
new_upload = files.upload()
new_file = list(new_upload.keys())[0]

client.vector_stores.files.upload_and_poll(
    vector_store_id=vector_store.id,
    file=open(new_file, "rb")
)

print("New paper added:", new_file)

Saving 06_Kim et al. (2023).pdf to 06_Kim et al. (2023).pdf
New paper added: 06_Kim et al. (2023).pdf


### Step 5. Ask the Same Question Again

Run the same question after adding the new paper and compare the result.

In [7]:
updated_response = client.responses.create(
    model=ANSWER_MODEL,
    input=QUESTION,
    tools=[
        {
            "type": "file_search",
            "vector_store_ids": [vector_store.id]
        }
    ]
)

print(updated_response.output_text)

## Comparison of the papers’ own empirical findings

| Paper | Environmental-performance measure and financial-performance measure | Paper’s own empirical finding |
|---|---|---|
| **Boakye et al. (2020)** | Multiple sustainable-environmental practices among UK listed SMEs; ROA and Tobin’s q | The relationship is generally **nonlinear**. Energy efficiency, GHG-related practices, and materials/resource efficiency show inverted-U-shaped relationships with financial performance: environmental engagement is beneficial at lower levels but becomes financially unfavorable beyond a point. For ROA, almost all practices are positively associated with performance except environmental compliance. For Tobin’s q, only waste management, materials/resource efficiency, and stakeholder engagement have significant positive relationships; energy efficiency, GHG practices, and compliance are not significant. Thus, environmental practices appear more consistently related to **internal accounting performance

### Step 6. Automate Structured Research Extraction

Apply the same extraction rules across all papers, organize the results as structured data, and save them as a CSV file.

In [8]:
import json
import pandas as pd

MATRIX_PROMPT = """
Using only the papers in the research knowledge base,
create a research evidence matrix.

Return JSON only as a list of records with these fields:

Paper
Country_Sample
Study_Period
Method
Environmental_Performance_Measure
Financial_Performance_Measure
Relationship
Main_Finding

Use the actual measures reported in each paper.
For Relationship, use:
Positive, Negative, Insignificant, Nonlinear, or Mixed.

Report each paper's own empirical findings, not prior studies
cited in the paper.

If information is unclear, write "Not reported".
Do not infer unsupported information.
"""

matrix_response = client.responses.create(
    model=ANSWER_MODEL,
    input=MATRIX_PROMPT,
    tools=[
        {
            "type": "file_search",
            "vector_store_ids": [vector_store.id]
        }
    ]
)

data = json.loads(matrix_response.output_text)
evidence_matrix = pd.DataFrame(data)

display(evidence_matrix)

evidence_matrix.to_csv(
    "Research_Evidence_Matrix.csv",
    index=False,
    encoding="utf-8-sig"
)

print("File created: Research_Evidence_Matrix.csv")

,Paper,Country_Sample,Study_Period,Method,Environmental_Performance_Measure,Financial_Performance_Measure,Relationship,Main_Finding
0,"Boakye et al. (2020), Sustainable environmenta...",United Kingdom; 201 SMEs listed on the Alterna...,2011–2016,Fixed-effects panel regression using OLS; two-...,Disaggregated sustainable environmental policy...,Return on assets (ROA) as the main accounting ...,Mixed,"Energy efficiency, greenhouse-gas practices, a..."
1,"Gallego-Alvarez, Segura, and Martínez-Ferrero ...",89 international companies from 21 countries,2006–2009,Panel-data regression with fixed-effects or ra...,"Variation in CO2 emissions, measured as the pe...",Return on equity (ROE); return on assets (ROA)...,Mixed,CO2-emission reduction had a significant posit...
2,"Fujii, Iwata, Kaneko, and Managi (2013), Corpo...",Japan; listed manufacturing firms on the Tokyo...,CO2 data: 2006–2008; toxic-chemical data: 2001...,Firm- and time-fixed-effects panel regressions...,Environmental efficiency measured as sales per...,"Return on assets (ROA), return on sales (ROS),...",Mixed,CO2-based environmental efficiency was positiv...
3,"Misani and Pogutz (2015), Unraveling the effec...",127 firms in carbon-intensive industries from ...,2007–2013,Hierarchical ordinary least squares regression...,Carbon performance: industry-adjusted Scope 1 ...,Tobin’s q as the main measure; return on equit...,Nonlinear,Carbon performance had an inverse U-shaped rel...
4,"Muhammad, Scrimgeour, Reddy, and Abidin (2015)...","Australia; publicly listed companies, with the...",2001–2010; pre-financial-crisis period 2001–20...,Panel regression controlling for unobserved co...,Corporate Environmental Performance (CEP) base...,"Return on assets (ROA), defined as EBIT divide...",Mixed,CEP was positively and significantly associate...
5,"Kim, Atukeren, and Kim (2023), Does the market...",South Korea; 156 firms classified as business-...,2011–2019,Firm-level panel regressions using B2B and B2C...,GHG emissions intensity measured as GHG emissi...,Tobin’s q; profit margin in the robustness ana...,Mixed,GHG emissions intensity was positively associa...


File created: Research_Evidence_Matrix.csv
